# Traigent Quickstart Demo Walkthrough

This notebook walks through the Traigent quickstart examples, demonstrating **zero-code LLM optimization using decorators**.

Instead of manually tuning hyperparameters, Traigent automatically explores configuration spaces and finds optimal settings for your specific use case.

## What You'll Learn

1. **Simple Q&A Optimization** - Basic decorator usage with multi-objective optimization
2. **RAG Optimization** - Production-ready retrieval with FAISS and chunking
3. **Custom Scorers** - Domain-specific evaluation functions
4. **Results Analysis** - Understanding what the optimization discovered

---

## Setup

First, let's install dependencies and set up the environment.

### Install Dependencies

Run the cell below to install required packages. Skip if already installed.

In [1]:
# Install dependencies (run once, then skip)
# Uncomment and run if you haven't installed these packages yet

# !pip install langchain-openai langchain-community langchain-text-splitters faiss-cpu jupyter ipykernel pandas numpy python-dotenv

# Or install all at once from requirements.txt:
# !pip install -r requirements.txt

# Verify installations
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import FAISS
    from langchain_openai import ChatOpenAI
    from dotenv import load_dotenv
    print("✓ All dependencies installed correctly!")
except ImportError as e:
    print(f"❌ Missing dependency: {e}")
    print("Please uncomment and run the pip install command above")

✓ All dependencies installed correctly!


In [2]:
import os
import sys
from pathlib import Path

# Ensure we're in the right directory
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Add to path if needed
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

QUICKSTART_DIR = REPO_ROOT / "examples" / "quickstart"
print(f"Repo root: {REPO_ROOT}")
print(f"Quickstart dir: {QUICKSTART_DIR}")

Repo root: REDACTED_TRAIGENT_ROOT/Traigent
Quickstart dir: REDACTED_TRAIGENT_ROOT/Traigent/examples/quickstart


In [3]:
# Load environment variables from .env file
from dotenv import load_dotenv

# Try multiple .env locations
env_paths = [
    REPO_ROOT / "walkthrough" / "examples" / "real" / ".env",  # Your .env location
    REPO_ROOT / ".env",
    QUICKSTART_DIR / ".env",
]

env_loaded = False
for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"✓ Loaded .env from: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("ℹ No .env file found, using environment variables")

# Check for API key (required for real LLM calls)
api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    print(f"✓ OPENAI_API_KEY is set ({api_key[:8]}...)")
else:
    print("⚠ OPENAI_API_KEY not set - examples will need it for real LLM calls")
    print("  Either create a .env file or run: export OPENAI_API_KEY='your-key-here'")

✓ Loaded .env from: REDACTED_TRAIGENT_ROOT/Traigent/walkthrough/examples/real/.env
✓ OPENAI_API_KEY is set (sk-proj-...)


---

## Part 1: Simple Q&A Optimization

**File**: `01_simple_qa_real_llm.py`

This example demonstrates basic Traigent optimization on a Q&A task with real OpenAI API calls.

### 1.1 The Configuration Space

Traigent explores a **configuration space** - all possible combinations of hyperparameters:

In [4]:
# Configuration space from the example
CONFIGURATION_SPACE = {
    "model": ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"],
    "temperature": [0.1, 0.3, 0.5, 0.7, 0.9],
    "max_tokens": [50, 100, 200],
}

# Calculate total configurations
total_configs = 1
for key, values in CONFIGURATION_SPACE.items():
    print(f"{key}: {len(values)} options -> {values}")
    total_configs *= len(values)

print(f"\nTotal possible configurations: {total_configs}")

model: 3 options -> ['gpt-3.5-turbo', 'gpt-4o-mini', 'gpt-4o']
temperature: 5 options -> [0.1, 0.3, 0.5, 0.7, 0.9]
max_tokens: 3 options -> [50, 100, 200]

Total possible configurations: 45


### 1.2 Constraints - Pruning Bad Configurations

Not all configurations make sense. Constraints skip invalid or wasteful combinations:

In [5]:
# Constraints from the example
constraints = [
    # Don't use high temperature with GPT-4o (expensive + unpredictable)
    lambda cfg: cfg["temperature"] <= 0.5 if cfg["model"] == "gpt-4o" else True,
    # Ensure max_tokens is reasonable for the model tier
    lambda cfg: cfg["max_tokens"] <= 100 if cfg["model"] == "gpt-3.5-turbo" else True,
]

# Let's see how many configs pass the constraints
from itertools import product

valid_configs = []
for model, temp, tokens in product(
    CONFIGURATION_SPACE["model"],
    CONFIGURATION_SPACE["temperature"],
    CONFIGURATION_SPACE["max_tokens"]
):
    cfg = {"model": model, "temperature": temp, "max_tokens": tokens}
    if all(constraint(cfg) for constraint in constraints):
        valid_configs.append(cfg)

print(f"Valid configurations after constraints: {len(valid_configs)} / {total_configs}")
print(f"Pruned: {total_configs - len(valid_configs)} invalid configs")
print("\nExample valid configs:")
for cfg in valid_configs[:5]:
    print(f"  {cfg}")

Valid configurations after constraints: 34 / 45
Pruned: 11 invalid configs

Example valid configs:
  {'model': 'gpt-3.5-turbo', 'temperature': 0.1, 'max_tokens': 50}
  {'model': 'gpt-3.5-turbo', 'temperature': 0.1, 'max_tokens': 100}
  {'model': 'gpt-3.5-turbo', 'temperature': 0.3, 'max_tokens': 50}
  {'model': 'gpt-3.5-turbo', 'temperature': 0.3, 'max_tokens': 100}
  {'model': 'gpt-3.5-turbo', 'temperature': 0.5, 'max_tokens': 50}


### 1.3 The @traigent.optimize Decorator

Here's the core pattern - a normal function wrapped with the `@traigent.optimize` decorator:

In [6]:
# This is the decorated function from the example (simplified for illustration)
example_code = '''
@traigent.optimize(
    configuration_space={
        "model": ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"],
        "temperature": [0.1, 0.3, 0.5, 0.7, 0.9],
        "max_tokens": [50, 100, 200],
    },
    objectives=["accuracy", "cost", "latency"],  # Multi-objective!
    constraints=[
        lambda cfg: cfg["temperature"] <= 0.5 if cfg["model"] == "gpt-4o" else True,
    ],
    metric_functions={"accuracy": custom_accuracy_scorer},
    eval_dataset="path/to/dataset.jsonl",
    max_trials=5,
)
def simple_qa_agent(question: str, model: str, temperature: float, max_tokens: int) -> str:
    """Traigent automatically injects optimized parameters."""
    llm = ChatOpenAI(model=model, temperature=temperature, max_tokens=max_tokens)
    return str(llm.invoke(f"Question: {question}\\nAnswer:").content)
'''
print(example_code)


@traigent.optimize(
    configuration_space={
        "model": ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"],
        "temperature": [0.1, 0.3, 0.5, 0.7, 0.9],
        "max_tokens": [50, 100, 200],
    },
    objectives=["accuracy", "cost", "latency"],  # Multi-objective!
    constraints=[
        lambda cfg: cfg["temperature"] <= 0.5 if cfg["model"] == "gpt-4o" else True,
    ],
    metric_functions={"accuracy": custom_accuracy_scorer},
    eval_dataset="path/to/dataset.jsonl",
    max_trials=5,
)
def simple_qa_agent(question: str, model: str, temperature: float, max_tokens: int) -> str:
    """Traigent automatically injects optimized parameters."""
    llm = ChatOpenAI(model=model, temperature=temperature, max_tokens=max_tokens)
    return str(llm.invoke(f"Question: {question}\nAnswer:").content)



### 1.4 Key Features

| Feature | What It Does |
|---------|-------------|
| `configuration_space` | Defines all hyperparameters to explore |
| `objectives` | Multi-objective: accuracy AND cost AND latency |
| `constraints` | Lambda functions to skip invalid configs |
| `metric_functions` | Custom scoring (not just string matching) |
| `eval_dataset` | JSONL file with input/output pairs |
| `max_trials` | Limit exploration for quick demos |

---

## Part 2: Customer Support RAG Optimization

**File**: `02_customer_support_rag_real_llm.py`

This example shows a production-ready RAG system with FAISS vector store and document chunking.

### 2.1 The RAG Architecture

```
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│  Knowledge  │────▶│   Chunking   │────▶│    FAISS    │
│    Base     │     │ (configurable)│     │  Vector DB  │
└─────────────┘     └──────────────┘     └─────────────┘
                                                │
                    ┌──────────────┐            │ similarity
                    │   Customer   │◀───────────┘  search
                    │    Query     │
                    └──────────────┘
                           │
                           ▼
                    ┌──────────────┐
                    │   LLM with   │──▶ Answer
                    │   Context    │
                    └──────────────┘
```

### 2.2 The Knowledge Base

Let's explore the knowledge base documents:

In [7]:
KNOWLEDGE_BASE_PATH = QUICKSTART_DIR / "knowledge_base"

print("Knowledge Base Structure:")
print("=" * 50)

for category_dir in sorted(KNOWLEDGE_BASE_PATH.iterdir()):
    if category_dir.is_dir():
        for file_path in sorted(category_dir.glob("*.md")):
            content = file_path.read_text()
            print(f"\n📁 {category_dir.name}/{file_path.name}")
            print(f"   {len(content):,} characters")
            # Show first 150 chars
            preview = content[:150].replace("\n", " ")
            print(f"   Preview: {preview}...")

Knowledge Base Structure:

📁 gift_cards/gift_cards.md
   1,441 characters
   Preview: # Gift Cards  ## About Our Gift Cards Give the gift of choice! Our gift cards are the perfect present for any occasion.  ## Gift Card Options - $25, $...

📁 hours/store_hours.md
   1,054 characters
   Preview: # Business Hours  ## Customer Support Hours  ### Phone and Live Chat Support - Monday through Friday: 9:00 AM - 5:00 PM EST - Saturday: 10:00 AM - 4:0...

📁 payments/payment_methods.md
   1,509 characters
   Preview: # Payment Methods  ## Accepted Payment Types We accept a wide variety of payment methods to make your shopping experience convenient and secure.  ### ...

📁 pricing/price_matching.md
   1,594 characters
   Preview: # Price Match Guarantee  ## Our Promise We're committed to offering competitive prices. If you find a lower price on an identical item from an authori...

📁 returns/returns_policy.md
   1,445 characters
   Preview: # Return Policy  ## Overview We want you to be completely

### 2.3 Sample Document: Returns Policy

In [20]:
returns_policy = (KNOWLEDGE_BASE_PATH / "returns" / "returns_policy.md").read_text()
print(returns_policy)

# Return Policy

## Overview
We want you to be completely satisfied with your purchase. If you're not happy with your order, we offer a comprehensive return policy to make things right.

## Return Window
Returns are accepted within 30 days of the original purchase date. Items must be unused and in their original packaging with all tags attached. After 30 days, we cannot accept returns but may offer store credit on a case-by-case basis.

## How to Initiate a Return
1. Log into your account and go to Order History
2. Select the order containing the item you wish to return
3. Click "Request Return" and select a reason
4. Print the prepaid shipping label (free for defective items)
5. Pack the item securely and drop it off at any carrier location

## Refund Processing
Once we receive your return, please allow 5-7 business days for inspection and processing. Refunds are issued to the original payment method. Credit card refunds may take an additional 3-5 business days to appear on your state

### 2.4 RAG-Specific Configuration Space

The RAG example adds parameters that traditional tuning often ignores:

In [21]:
RAG_CONFIGURATION_SPACE = {
    "model": ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"],
    "temperature": [0.1, 0.3, 0.5, 0.7],
    "k": [2, 3, 5],              # Number of chunks to retrieve
    "chunk_size": [300, 500, 800],    # Characters per chunk
    "chunk_overlap": [25, 50, 100],   # Sliding window overlap
}

print("RAG Configuration Space:")
print("=" * 50)
for key, values in RAG_CONFIGURATION_SPACE.items():
    print(f"{key:15} : {values}")

total = 1
for v in RAG_CONFIGURATION_SPACE.values():
    total *= len(v)
print(f"\nTotal combinations: {total}")

RAG Configuration Space:
model           : ['gpt-3.5-turbo', 'gpt-4o-mini', 'gpt-4o']
temperature     : [0.1, 0.3, 0.5, 0.7]
k               : [2, 3, 5]
chunk_size      : [300, 500, 800]
chunk_overlap   : [25, 50, 100]

Total combinations: 324


### 2.5 Why These Parameters Matter

| Parameter | Effect |
|-----------|--------|
| **k** | More chunks = more context, but also more noise |
| **chunk_size** | Smaller = precise retrieval, Larger = more context per chunk |
| **chunk_overlap** | Preserves context at chunk boundaries |

### 2.6 Document Chunking Demonstration

In [22]:
# Simulate chunking with different sizes
def simple_chunk(text: str, chunk_size: int, overlap: int) -> list:
    """Simple chunking for demonstration."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

# Chunk the returns policy with different settings
print(f"Original document: {len(returns_policy):,} characters")
print("=" * 50)

for chunk_size in [300, 500, 800]:
    chunks = simple_chunk(returns_policy, chunk_size, 50)
    print(f"\nchunk_size={chunk_size}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks[:2]):
        preview = chunk[:60].replace("\n", " ")
        print(f"  Chunk {i+1}: \"{preview}...\"")

Original document: 1,445 characters

chunk_size=300: 6 chunks
  Chunk 1: "# Return Policy  ## Overview We want you to be completely sa..."
  Chunk 2: "ginal purchase date. Items must be unused and in their origi..."

chunk_size=500: 4 chunks
  Chunk 1: "# Return Policy  ## Overview We want you to be completely sa..."
  Chunk 2: " Initiate a Return 1. Log into your account and go to Order ..."

chunk_size=800: 2 chunks
  Chunk 1: "# Return Policy  ## Overview We want you to be completely sa..."
  Chunk 2: "n  ## Refund Processing Once we receive your return, please ..."


### 2.7 The Evaluation Dataset

In [23]:
import json

rag_dataset_path = QUICKSTART_DIR / "rag_feedback.jsonl"

if rag_dataset_path.exists():
    questions = []
    with open(rag_dataset_path) as f:
        for line in f:
            if line.strip():
                questions.append(json.loads(line))
    
    print(f"RAG Evaluation Dataset: {len(questions)} questions")
    print("=" * 60)
    
    for i, q in enumerate(questions[:5], 1):
        query = q["input"]["query"]
        expected = q["output"]
        print(f"\n{i}. Q: {query}")
        print(f"   Expected: {expected}")
    
    if len(questions) > 5:
        print(f"\n... and {len(questions) - 5} more questions")
else:
    print(f"Dataset not found at {rag_dataset_path}")

RAG Evaluation Dataset: 21 questions

1. Q: If I return an item I paid for with a gift card, how do I get my money back?
   Expected: Refunded back to a gift card

2. Q: I ordered at 3pm on a weekday - when will it ship and how long until it arrives with standard shipping?
   Expected: Ships next business day, then 5-7 business days for delivery

3. Q: I want to buy a physical gift card for someone in Canada - can I ship it there and how much will it cost?
   Expected: Yes, physical cards ship free domestically, international shipping rates calculated at checkout

4. Q: If I order at 1pm on Saturday and choose overnight shipping, when will it arrive?
   Expected: Weekend orders ship Monday, overnight delivery means it arrives Tuesday

5. Q: I paid with PayPal and returned an item - when will I see the refund?
   Expected: 5-7 business days after inspection, refund issued to original payment method

... and 16 more questions


---

## Part 3: Custom Scorers

**File**: `scorers.py`

Custom scorers enable domain-specific evaluation beyond simple string matching.

In [24]:
# Load and display the scorer code
scorers_path = QUICKSTART_DIR / "scorers.py"
if scorers_path.exists():
    print(scorers_path.read_text())

"""Shared scoring functions for TraiGent quickstart examples."""

import re


def custom_accuracy_scorer(
    output: str, expected: str, llm_metrics: dict = None
) -> float:
    """Custom scoring function that checks if expected answer is in output.

    For single-letter answers (A-J), looks for patterns like "A.", "A)", "(A)",
    "Answer: A", "answer is A" to avoid false positives from letters in words.

    For other answers, uses case-insensitive containment check.

    Args:
        output: The LLM's response
        expected: The expected answer from the dataset
        llm_metrics: Additional metrics from the LLM call (tokens, latency, etc.)

    Returns:
        Score between 0 and 1
    """
    if not output or not expected:
        return 0.0

    expected = expected.strip()
    output_lower = output.lower()
    expected_lower = expected.lower()

    # For single-letter answers (multiple choice), use strict matching
    if len(expected) == 1 and expected.upper() in "ABCDEFG

### 3.1 Why Custom Scorers Matter

LLMs often wrap answers in explanations. The scorer needs to extract the actual answer:

In [25]:
import re

def custom_accuracy_scorer(output: str, expected: str) -> float:
    """Custom scoring function from scorers.py"""
    if not output or not expected:
        return 0.0

    expected = expected.strip()
    output_lower = output.lower()
    expected_lower = expected.lower()

    # For single-letter answers (multiple choice)
    if len(expected) == 1 and expected.upper() in "ABCDEFGHIJ":
        letter = expected.upper()
        patterns = [
            rf"\b{letter}\.",
            rf"\({letter}\)",
            rf"answer[:\s]+{letter}\b",
        ]
        for pattern in patterns:
            if re.search(pattern, output, re.IGNORECASE):
                return 1.0
        return 0.0

    # For yes/no/true/false
    if expected_lower in ["yes", "no", "true", "false"]:
        return 1.0 if re.search(rf"\b{expected_lower}\b", output_lower) else 0.0

    # Default: containment
    return 1.0 if expected_lower in output_lower else 0.0


# Test the scorer
test_cases = [
    ("The answer is A. Because the sky is blue.", "A", "Multiple choice"),
    ("I believe (B) is correct since...", "B", "Multiple choice with parens"),
    ("A car drove by", "A", "False positive avoided"),
    ("Yes, that's correct!", "yes", "Yes/No"),
    ("The refund takes 30 days", "30 days", "Containment"),
]

print("Scorer Test Cases:")
print("=" * 60)
for output, expected, description in test_cases:
    score = custom_accuracy_scorer(output, expected)
    status = "✓" if score > 0 else "✗"
    print(f"{status} {description}")
    print(f"  Output: \"{output}\"")
    print(f"  Expected: \"{expected}\" -> Score: {score}")
    print()

Scorer Test Cases:
✓ Multiple choice
  Output: "The answer is A. Because the sky is blue."
  Expected: "A" -> Score: 1.0

✓ Multiple choice with parens
  Output: "I believe (B) is correct since..."
  Expected: "B" -> Score: 1.0

✗ False positive avoided
  Output: "A car drove by"
  Expected: "A" -> Score: 0.0

✓ Yes/No
  Output: "Yes, that's correct!"
  Expected: "yes" -> Score: 1.0

✓ Containment
  Output: "The refund takes 30 days"
  Expected: "30 days" -> Score: 1.0



### 3.2 LLM-as-Judge (RAG Example)

The RAG example uses a more sophisticated scorer - an LLM judges semantic equivalence:

In [ ]:
llm_judge_code = '''
def rag_accuracy_scorer(output: str, expected: str) -> float:
    """Score RAG response quality using LLM-as-Judge."""
    judge = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=10)

    prompt = f"""You are evaluating if an AI assistant's answer is correct.

Expected answer: {expected}
Actual answer: {output}

Does the actual answer correctly convey the same key information?
- Minor wording differences are OK
- Extra details are OK
- Core facts must match

Reply with only "YES" or "NO"."""

    response = judge.invoke(prompt)
    return 1.0 if "YES" in str(response.content).upper() else 0.0
'''
print(llm_judge_code)

---

## Part 4: Optimization Results Analysis

Let's analyze what the optimization discovered.

### 4.1 Simple Q&A Results

In [ ]:
results_dir = QUICKSTART_DIR / "results"

# Load Simple Q&A best config
qa_best_config_path = results_dir / "best_config.json"
if qa_best_config_path.exists():
    qa_best = json.loads(qa_best_config_path.read_text())
    print("Simple Q&A Optimization Results")
    print("=" * 40)
    print(f"Best Score: {qa_best['best_score']:.1%}")
    print(f"Best Config:")
    for key, value in qa_best['best_config'].items():
        print(f"  {key}: {value}")
else:
    print("No results found - run the optimization first")

### 4.2 RAG Optimization Results

In [ ]:
# Load RAG best config
rag_best_config_path = results_dir / "rag_best_config.json"
if rag_best_config_path.exists():
    rag_best = json.loads(rag_best_config_path.read_text())
    print("RAG Optimization Results")
    print("=" * 40)
    print(f"Best Score: {rag_best['best_score']:.1%}")
    print(f"Best Config:")
    for key, value in rag_best['best_config'].items():
        print(f"  {key}: {value}")
else:
    print("No results found - run the optimization first")

### 4.3 Detailed RAG Analysis

The optimization run produced detailed analysis. Key findings:

In [ ]:
# Summary from the detailed analysis
analysis = {
    "Dataset Evolution": {
        "v1 (original)": {"questions": 30, "best_score": "80.0%"},
        "v2 (balanced)": {"questions": 28, "best_score": "64.3%"},
        "v3 (refined)": {"questions": 21, "best_score": "85.7%"},
    },
    "Top Configurations (tied at 85.7%)": [
        {"model": "gpt-3.5-turbo", "k": 3, "chunk_size": 500},
        {"model": "gpt-3.5-turbo", "k": 5, "chunk_size": 800},
        {"model": "gpt-4o", "k": 2, "chunk_size": 300},
    ],
    "Accuracy by Model": {
        "gpt-3.5-turbo": "85.7% (best, AND cheapest!)",
        "gpt-4o-mini": "76.2%",
        "gpt-4o": "81.0% avg, 85.7% peak",
    },
    "Systematic Failures": [
        "Numerical reasoning (e.g., 2:30pm after 12pm cutoff)",
        "Multi-constraint synthesis (PO Box + Canada + overnight)",
    ]
}

print("Detailed RAG Analysis Summary")
print("=" * 60)

print("\n📊 Dataset Evolution:")
for version, stats in analysis["Dataset Evolution"].items():
    print(f"  {version}: {stats['questions']} questions -> {stats['best_score']}")

print("\n🏆 Top Configurations (all tied at 85.7%):")
for cfg in analysis["Top Configurations (tied at 85.7%)"]:
    print(f"  {cfg}")

print("\n📈 Accuracy by Model:")
for model, acc in analysis["Accuracy by Model"].items():
    print(f"  {model}: {acc}")

print("\n⚠️  Systematic Failures (all configs fail):")
for failure in analysis["Systematic Failures"]:
    print(f"  • {failure}")

### 4.4 The "Aha" Moment 💡

**The cheaper model (gpt-3.5-turbo) outperformed gpt-4o on this task!**

Without systematic optimization, most developers would default to the "best" model and overpay for worse results.

In [ ]:
# Cost comparison (approximate)
costs = {
    "gpt-3.5-turbo": 0.002,  # per 1K tokens
    "gpt-4o-mini": 0.00015,
    "gpt-4o": 0.005,
}

print("Cost vs Accuracy Analysis")
print("=" * 50)
print(f"{'Model':<20} {'Accuracy':<15} {'Cost/1K tokens'}")
print("-" * 50)
print(f"{'gpt-3.5-turbo':<20} {'85.7% ⭐':<15} ${costs['gpt-3.5-turbo']:.4f}")
print(f"{'gpt-4o-mini':<20} {'76.2%':<15} ${costs['gpt-4o-mini']:.5f}")
print(f"{'gpt-4o':<20} {'81.0% avg':<15} ${costs['gpt-4o']:.4f}")
print()
print("💰 Insight: gpt-3.5-turbo is BOTH the most accurate AND cheaper than gpt-4o!")

---

## Part 5: Traigent's Value Proposition

In [ ]:
comparison = [
    ("Model selection", "Manual A/B testing", "Automatic exploration of 45+ configs"),
    ("Temperature tuning", "Guess-and-check", "Data-driven optimization"),
    ("RAG parameters", "Often ignored (k, chunk_size)", "Optimize the full stack"),
    ("Metrics", "Single metric (accuracy)", "Multi-objective (accuracy + cost + latency)"),
    ("Bad configs", "Waste time/money testing", "Smart constraint pruning"),
]

print("Traditional Approach vs Traigent")
print("=" * 80)
print(f"{'Aspect':<20} {'Traditional':<30} {'With Traigent'}")
print("-" * 80)
for aspect, traditional, traigent in comparison:
    print(f"{aspect:<20} {traditional:<30} {traigent}")

### Zero-Code Integration

The developer writes a **normal function**. Traigent adds the decorator and handles everything else:

```python
@traigent.optimize(...)
def my_agent(query: str, model: str, temperature: float) -> str:
    # Your normal code here
    return response

# Run optimization
results = await my_agent.optimize()

# Use the best config
best_config = my_agent.get_best_config()
my_agent.set_config(best_config)
```

---

## Summary

These quickstart examples demonstrate Traigent's ability to:

1. ✅ **Define rich configuration spaces** (model, temperature, RAG params)
2. ✅ **Apply intelligent constraints** to prune invalid configurations
3. ✅ **Use custom scorers** (pattern matching or LLM-as-Judge)
4. ✅ **Discover non-obvious optima** (cheaper model = better results!)
5. ✅ **Provide detailed analytics** on per-parameter performance

### Key Insight

The optimization discovered that **gpt-3.5-turbo outperforms gpt-4o** on the RAG task - a non-obvious result that saves money while improving accuracy.

---

## Next Steps

To run the actual optimization:

```bash
# Set your API key
export OPENAI_API_KEY='your-key-here'

# Run Simple Q&A optimization
python examples/quickstart/01_simple_qa_real_llm.py

# Run RAG optimization (requires faiss-cpu)
pip install faiss-cpu
python examples/quickstart/02_customer_support_rag_real_llm.py
```